# 02 · Modeling, Tuning & Explainability
**Mobile Price Prediction — FYP**

Trains and compares the six classifiers, tunes the gradient-boosting models with Optuna, evaluates on a held-out test set, and produces SHAP explanations (proposal §7–§9).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # make `src` importable
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 40)

## 1. Run the full training pipeline
This trains every model, runs Optuna on XGBoost/LightGBM, saves the best artifacts, and writes `reports/metrics.json`.  
*(Use `main(fast=True)` for a quick run with fewer trials.)*

In [ ]:
from src.models.train import main
report = main(fast=False)

## 2. Model comparison table

In [ ]:
import json
report = json.load(open('../reports/metrics.json'))
rows = []
for name, r in report['classification'].items():
    rows.append({'model': name,
                 'cv_f1_macro': round(r['cv']['f1_macro_mean'],4),
                 'test_accuracy': round(r['test']['accuracy'],4),
                 'test_f1_macro': round(r['test']['f1_macro'],4),
                 'roc_auc_ovr': round(r['test'].get('roc_auc_ovr') or 0,4)})
pd.DataFrame(rows).sort_values('test_f1_macro', ascending=False)

## 3. Regression results

In [ ]:
pd.DataFrame(report['regression']).T.round(4)

## 4. SHAP feature importance
Computed on the best tree model (proposal Fig. 4).

In [ ]:
pd.DataFrame(report['shap_top_features'],
             columns=['feature','mean_abs_shap']).head(10)

In [ ]:
from IPython.display import Image
Image('../reports/figures/09_shap_importance.png')

## 5. Try a single prediction

In [ ]:
from src.models.predict import get_service, EXAMPLE_PHONE
get_service().predict(EXAMPLE_PHONE)

## 6. Acceptance targets
Classification accuracy ≥ 93 %, F1-macro ≥ 0.92, regression R² ≥ 0.90, MAPE < 12 % (proposal §12.1).

In [ ]:
best = report['classification'][report['best_classifier']]['test']
reg  = report['regression'][report['best_regressor']]
print('accuracy :', round(best['accuracy'],4))
print('f1_macro :', round(best['f1_macro'],4))
print('reg R2   :', round(reg['r2'],4))
print('reg MAPE :', round(reg['mape'],4))